# Generación de datos para INSERT

Lee los CSV brutos y construye un DataFrame por cada tabla de la BD,
generadon los INSERTS de sql.

In [23]:
import pandas as pd
import numpy as np

DS_PROYECTOS = ['HLF', 'EDA', 'BBDD', 'ML', 'Deployment']
FS_PROYECTOS = ['WebDev', 'FrontEnd', 'Backend', 'React', 'FullStack']

# Serializa un valor Python a literal SQL: NULL, entero sin comillas, o string con comillas simples escapadas
def fmt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return 'NULL'
    if isinstance(v, (int, np.integer)): return str(int(v))
    return "'" + str(v).replace("'", "''") + "'"

# Crea el INSERT en SQL
def print_insert(tabla, df, cols):
    rows = ['    (' + ', '.join(fmt(row[c]) for c in cols) + ')' for _, row in df.iterrows()]
    print(f'INSERT INTO {tabla} ({", ".join(cols)}) VALUES')
    print(',\n'.join(rows) + ';\n')

## 1. Lectura de CSVs

In [24]:
clase1   = pd.read_csv('../src/datos_brutos/clase_1.csv', sep=';')
clase2   = pd.read_csv('../src/datos_brutos/clase_2.csv', sep=';')
clase3   = pd.read_csv('../src/datos_brutos/clase_3.csv', sep=';')
clase4   = pd.read_csv('../src/datos_brutos/clase_4.csv', sep=';')
claustro = pd.read_csv('../src/datos_brutos/claustro.csv', sep=';')

DS_RENAME = {f'Proyecto_{p}': p for p in DS_PROYECTOS}
FS_RENAME = {f'Proyecto_{p}': p for p in FS_PROYECTOS}
FS_RENAME['Proyecto_FullSatck'] = 'FullStack'  # typo en el CSV original

for df, rename_map, vert in [
    (clase1, DS_RENAME, 'DataScience'), (clase2, DS_RENAME, 'DataScience'),
    (clase3, FS_RENAME, 'FullStack'),   (clase4, FS_RENAME, 'FullStack'),
]:
    df.rename(columns={**rename_map, 'Promoción': 'Promocion'}, inplace=True)
    df['Vertical'] = vert

all_estudiante = pd.concat([clase1, clase2, clase3, clase4], ignore_index=True)
all_estudiante['Fecha_comienzo'] = pd.to_datetime(all_estudiante['Fecha_comienzo'], format='%d/%m/%Y')
all_estudiante['Anio'] = all_estudiante['Fecha_comienzo'].dt.year

claustro['Rol']      = claustro['Rol'].map({'TA': 'Teach Assistant', 'LI': 'Lead Instructor'})
claustro['Vertical'] = claustro['Vertical'].map({'DS': 'DataScience', 'FS': 'FullStack'})
_year_map            = all_estudiante.groupby('Promocion')['Anio'].first().to_dict()
claustro['Anio']     = claustro['Promocion'].map(_year_map)

all_estudiante.head(5)

,Nombre,Email,Promocion,Fecha_comienzo,Campus,HLF,EDA,BBDD,ML,Deployment,Vertical,WebDev,FrontEnd,Backend,React,FullStack,Anio
0,Jafet Casals,Jafet_Casals@gmail.com,Septiembre,2023-09-18,Madrid,Apto,No Apto,Apto,Apto,Apto,DataScience,NaN,NaN,NaN,NaN,NaN,2023
1,Jorge Manzanares,Jorge_Manzanares@gmail.com,Septiembre,2023-09-18,Madrid,Apto,No Apto,Apto,Apto,Apto,DataScience,NaN,NaN,NaN,NaN,NaN,2023
2,Onofre Adadia,Onofre_Adadia@gmail.com,Septiembre,2023-09-18,Madrid,Apto,Apto,Apto,No Apto,Apto,DataScience,NaN,NaN,NaN,NaN,NaN,2023
3,Merche Prada,Merche_Prada@gmail.com,Septiembre,2023-09-18,Madrid,Apto,No Apto,No Apto,Apto,No Apto,DataScience,NaN,NaN,NaN,NaN,NaN,2023
4,Pilar Abella,Pilar_Abella@gmail.com,Septiembre,2023-09-18,Madrid,Apto,No Apto,Apto,Apto,Apto,DataScience,NaN,NaN,NaN,NaN,NaN,2023


## 2. Tablas de catálogo

In [25]:
def make_lookup(values, id_col, nombre_col='nombre'):
    uniq = list(dict.fromkeys(values))
    return pd.DataFrame({id_col: range(1, len(uniq) + 1), nombre_col: uniq})

campus    = make_lookup(all_estudiante['Campus'].unique(),    'campus_id',    'campus')
vertical  = make_lookup(['DataScience', 'FullStack'],         'vertical_id',  'vertical')
modalidad = make_lookup(claustro['Modalidad'].unique(),       'modalidad_id', 'modalidad')
rol       = make_lookup(claustro['Rol'].unique(),             'rol_id',       'rol')

# promocion: clave compuesta (nombre, anio) para distinguir cohortes homónimas
promocion = (
    all_estudiante[['Promocion', 'Anio']]
    .drop_duplicates()
    .sort_values(['Anio', 'Promocion'])
    .reset_index(drop=True)
    .rename(columns={'Promocion': 'nombre', 'Anio': 'anio'})
)
promocion.insert(0, 'promocion_id', range(1, len(promocion) + 1))

for df in [campus, vertical, promocion, modalidad, rol]:
    display(df)

print_insert('campus',    campus,    ['campus'])
print_insert('vertical',  vertical,  ['vertical'])
print_insert('promocion', promocion, ['nombre', 'anio'])
print_insert('modalidad', modalidad, ['modalidad'])
print_insert('rol',       rol,       ['rol'])

,campus_id,campus
0,1,Madrid
1,2,Valencia


,vertical_id,vertical
0,1,DataScience
1,2,FullStack


,promocion_id,nombre,anio
0,1,Septiembre,2023
1,2,Febrero,2024


,modalidad_id,modalidad
0,1,Presencial
1,2,Online


,rol_id,rol
0,1,Teach Assistant
1,2,Lead Instructor


INSERT INTO campus (campus) VALUES
    ('Madrid'),
    ('Valencia');

INSERT INTO vertical (vertical) VALUES
    ('DataScience'),
    ('FullStack');

INSERT INTO promocion (nombre, anio) VALUES
    ('Septiembre', 2023),
    ('Febrero', 2024);

INSERT INTO modalidad (modalidad) VALUES
    ('Presencial'),
    ('Online');

INSERT INTO rol (rol) VALUES
    ('Teach Assistant'),
    ('Lead Instructor');



## 3. Proyecto_tipo

In [26]:
proyecto_tipo = pd.DataFrame(
    [(p, 'DataScience') for p in DS_PROYECTOS] + [(p, 'FullStack') for p in FS_PROYECTOS],
    columns=['proyecto', 'vert']
).merge(vertical, left_on='vert', right_on='vertical', suffixes=('', '_drop'))

proyecto_tipo = proyecto_tipo[['proyecto', 'vertical_id']].reset_index(drop=True)
proyecto_tipo.insert(0, 'proyecto_tipo_id', range(1, len(proyecto_tipo) + 1))

display(proyecto_tipo)
print_insert('proyecto_tipo', proyecto_tipo, ['proyecto', 'vertical_id'])

,proyecto_tipo_id,proyecto,vertical_id
0,1,HLF,1
1,2,EDA,1
2,3,BBDD,1
3,4,ML,1
4,5,Deployment,1
5,6,WebDev,2
6,7,FrontEnd,2
7,8,Backend,2
8,9,React,2
9,10,FullStack,2


INSERT INTO proyecto_tipo (proyecto, vertical_id) VALUES
    ('HLF', 1),
    ('EDA', 1),
    ('BBDD', 1),
    ('ML', 1),
    ('Deployment', 1),
    ('WebDev', 2),
    ('FrontEnd', 2),
    ('Backend', 2),
    ('React', 2),
    ('FullStack', 2);



## 4. Grupo

Los grupos se derivan únicamente del `claustro.csv`, que es la única fuente con el campo `Modalidad`.
La `fecha_inicio` se obtiene mapeando la promoción a partir de los datos de estudiantes.

In [29]:
# Mapa (Promocion, Anio) → fecha_inicio
_fecha_map = all_estudiante.groupby(['Promocion', 'Anio'])['Fecha_comienzo'].first().to_dict()

# Combos base del claustro
claustro_combos = claustro[['Campus', 'Vertical', 'Promocion', 'Modalidad', 'Anio']].drop_duplicates()

# Cohorts de estudiantes sin ninguna entrada en claustro → Presencial por defecto
_student_cohorts = all_estudiante[['Campus', 'Vertical', 'Promocion', 'Anio']].drop_duplicates()
_missing = (
    _student_cohorts
    .merge(claustro_combos[['Campus', 'Vertical', 'Promocion', 'Anio']].drop_duplicates(),
           on=['Campus', 'Vertical', 'Promocion', 'Anio'], how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns='_merge')
    .assign(Modalidad='Presencial')
)
grupo_combos = pd.concat([claustro_combos, _missing], ignore_index=True)

grupo = (
    grupo_combos
    .assign(Fecha_comienzo=lambda df: [_fecha_map.get((p, a)) for p, a in zip(df['Promocion'], df['Anio'])])
    .merge(campus,    left_on='Campus',             right_on='campus')
    .merge(vertical,  left_on='Vertical',           right_on='vertical')
    .merge(promocion, left_on=['Promocion', 'Anio'], right_on=['nombre', 'anio'])
    .merge(modalidad, left_on='Modalidad',           right_on='modalidad')
    .sort_values(['vertical_id', 'campus_id', 'promocion_id', 'modalidad_id'])
    .reset_index(drop=True)
)
grupo.insert(0, 'grupo_id', range(1, len(grupo) + 1))
grupo = grupo[['grupo_id', 'campus_id', 'vertical_id', 'promocion_id', 'modalidad_id', 'Fecha_comienzo']]
grupo = grupo.rename(columns={'Fecha_comienzo': 'fecha_inicio'})

display(grupo)
print_insert('grupo', grupo, ['campus_id', 'vertical_id', 'promocion_id', 'modalidad_id', 'fecha_inicio'])

,grupo_id,campus_id,vertical_id,promocion_id,modalidad_id,fecha_inicio
0,1,1,1,1,1,2023-09-18
1,2,1,1,1,2,2023-09-18
2,3,1,1,2,1,2024-02-12
3,4,1,2,1,1,2023-09-18
4,5,1,2,1,2,2023-09-18
5,6,1,2,2,1,2024-02-12
6,7,2,2,1,1,2023-09-18
7,8,2,2,2,1,2024-02-12
8,9,2,2,2,2,2024-02-12


INSERT INTO grupo (campus_id, vertical_id, promocion_id, modalidad_id, fecha_inicio) VALUES
    (1, 1, 1, 1, '2023-09-18 00:00:00'),
    (1, 1, 1, 2, '2023-09-18 00:00:00'),
    (1, 1, 2, 1, '2024-02-12 00:00:00'),
    (1, 2, 1, 1, '2023-09-18 00:00:00'),
    (1, 2, 1, 2, '2023-09-18 00:00:00'),
    (1, 2, 2, 1, '2024-02-12 00:00:00'),
    (2, 2, 1, 1, '2023-09-18 00:00:00'),
    (2, 2, 2, 1, '2024-02-12 00:00:00'),
    (2, 2, 2, 2, '2024-02-12 00:00:00');



## 5. Estudiante

In [30]:
li_modal = (
    claustro[claustro['Rol'] == 'Lead Instructor']
    [['Campus', 'Vertical', 'Promocion', 'Anio', 'Modalidad']]
    .drop_duplicates(subset=['Campus', 'Vertical', 'Promocion', 'Anio'])
    .rename(columns={'Modalidad': 'Modalidad_LI'})
)
ta_modal = (
    claustro[claustro['Rol'] == 'Teach Assistant']
    [['Campus', 'Vertical', 'Promocion', 'Anio', 'Modalidad']]
    .drop_duplicates(subset=['Campus', 'Vertical', 'Promocion', 'Anio'])
    .rename(columns={'Modalidad': 'Modalidad_TA'})
)

modal_estudiante = (
    ta_modal
    .merge(li_modal, on=['Campus', 'Vertical', 'Promocion', 'Anio'], how='outer')
    .assign(Modalidad=lambda df: df['Modalidad_LI'].fillna(df['Modalidad_TA']).fillna('Presencial'))
    [['Campus', 'Vertical', 'Promocion', 'Anio', 'Modalidad']]
)

estudiante_merged = (
    all_estudiante
    .merge(modal_estudiante, on=['Campus', 'Vertical', 'Promocion', 'Anio'], how='left')
    .assign(Modalidad=lambda df: df['Modalidad'].fillna('Presencial'))
    .merge(campus,    left_on='Campus',             right_on='campus')
    .merge(vertical,  left_on='Vertical',           right_on='vertical')
    .merge(promocion, left_on=['Promocion', 'Anio'], right_on=['nombre', 'anio'])
    .merge(modalidad, left_on='Modalidad',           right_on='modalidad')
    .merge(
        grupo[['grupo_id', 'campus_id', 'vertical_id', 'promocion_id', 'modalidad_id']],
        on=['campus_id', 'vertical_id', 'promocion_id', 'modalidad_id']
    )
)

estudiante = (
    estudiante_merged[['Nombre', 'Email', 'grupo_id']]
    .rename(columns={'Nombre': 'nombre', 'Email': 'email'})
    .reset_index(drop=True)
)
estudiante.insert(0, 'estudiante_id', range(1, len(estudiante) + 1))

display(estudiante.head(5))
print_insert('estudiante', estudiante, ['nombre', 'email', 'grupo_id'])

,estudiante_id,nombre,email,grupo_id
0,1,Jafet Casals,Jafet_Casals@gmail.com,2
1,2,Jorge Manzanares,Jorge_Manzanares@gmail.com,2
2,3,Onofre Adadia,Onofre_Adadia@gmail.com,2
3,4,Merche Prada,Merche_Prada@gmail.com,2
4,5,Pilar Abella,Pilar_Abella@gmail.com,2


INSERT INTO estudiante (nombre, email, grupo_id) VALUES
    ('Jafet Casals', 'Jafet_Casals@gmail.com', 2),
    ('Jorge Manzanares', 'Jorge_Manzanares@gmail.com', 2),
    ('Onofre Adadia', 'Onofre_Adadia@gmail.com', 2),
    ('Merche Prada', 'Merche_Prada@gmail.com', 2),
    ('Pilar Abella', 'Pilar_Abella@gmail.com', 2),
    ('Leoncio Tena', 'Leoncio_Tena@gmail.com', 2),
    ('Odalys Torrijos', 'Odalys_Torrijos@gmail.com', 2),
    ('Eduardo Caparrós', 'Eduardo_Caparrós@gmail.com', 2),
    ('Ignacio Goicoechea', 'Ignacio_Goicoechea@gmail.com', 2),
    ('Clementina Santos', 'Clementina_Santos@gmail.com', 2),
    ('Daniela Falcó', 'Daniela_Falcó@gmail.com', 2),
    ('Abraham Vélez', 'Abraham_Vélez@gmail.com', 2),
    ('Maximiliano Menéndez', 'Maximiliano_Menéndez@gmail.com', 2),
    ('Anita Heredia', 'Anita_Heredia@gmail.com', 2),
    ('Eli Casas', 'Eli_Casas@gmail.com', 2),
    ('Guillermo Borrego', 'Guillermo_Borrego@gmail.com', 3),
    ('Sergio Aguirre', 'Sergio_Aguirre@gmail.com', 3),
 

## 6. Calificacion

`melt` convierte las columnas de proyecto en filas.

In [ ]:
partes = []
for vert, proyecto in [('DataScience', DS_PROYECTOS), ('FullStack', FS_PROYECTOS)]:
    subset = (
        estudiante_merged[estudiante_merged['Vertical'] == vert]
        .merge(estudiante[['estudiante_id', 'email']], left_on='Email', right_on='email')
    )
    partes.append(
        subset.melt(
            id_vars=['estudiante_id'],
            value_vars=proyecto,
            var_name='proyecto',
            value_name='resultado'
        )
    )

calificacion = (
    pd.concat(partes, ignore_index=True)
    .merge(proyecto_tipo, left_on='proyecto', right_on='proyecto')
    .sort_values(['estudiante_id', 'proyecto_tipo_id'])
    [['estudiante_id', 'proyecto_tipo_id', 'resultado']]
    .reset_index(drop=True)
)
calificacion.insert(0, 'calificacion_id', range(1, len(calificacion) + 1))

display(calificacion.head(5))
print_insert('calificacion', calificacion, ['estudiante_id', 'proyecto_tipo_id', 'resultado'])

,calificacion_id,estudiante_id,proyecto_tipo_id,resultado
0,1,1,1,Apto
1,2,1,2,No Apto
2,3,1,3,Apto
3,4,1,4,Apto
4,5,1,5,Apto


INSERT INTO calificacion (estudiante_id, proyecto_tipo_id, resultado) VALUES
    (1, 1, 'Apto'),
    (1, 2, 'No Apto'),
    (1, 3, 'Apto'),
    (1, 4, 'Apto'),
    (1, 5, 'Apto'),
    (2, 1, 'Apto'),
    (2, 2, 'No Apto'),
    (2, 3, 'Apto'),
    (2, 4, 'Apto'),
    (2, 5, 'Apto'),
    (3, 1, 'Apto'),
    (3, 2, 'Apto'),
    (3, 3, 'Apto'),
    (3, 4, 'No Apto'),
    (3, 5, 'Apto'),
    (4, 1, 'Apto'),
    (4, 2, 'No Apto'),
    (4, 3, 'No Apto'),
    (4, 4, 'Apto'),
    (4, 5, 'No Apto'),
    (5, 1, 'Apto'),
    (5, 2, 'No Apto'),
    (5, 3, 'Apto'),
    (5, 4, 'Apto'),
    (5, 5, 'Apto'),
    (6, 1, 'Apto'),
    (6, 2, 'Apto'),
    (6, 3, 'Apto'),
    (6, 4, 'Apto'),
    (6, 5, 'Apto'),
    (7, 1, 'No Apto'),
    (7, 2, 'Apto'),
    (7, 3, 'Apto'),
    (7, 4, 'Apto'),
    (7, 5, 'Apto'),
    (8, 1, 'No Apto'),
    (8, 2, 'Apto'),
    (8, 3, 'Apto'),
    (8, 4, 'Apto'),
    (8, 5, 'Apto'),
    (9, 1, 'Apto'),
    (9, 2, 'Apto'),
    (9, 3, 'Apto'),
    (9, 4, 'No Apto'),
    (9, 5, 'A

## 7. profesor

In [ ]:
# Un profesor puede aparecer varias veces en claustro (un registro por grupo que imparte);
# drop_duplicates garantiza una sola fila por persona en la tabla profesor
profesor = (
    claustro
    .merge(rol, left_on='Rol', right_on='rol')
    [['Nombre', 'rol_id']]
    .rename(columns={'Nombre': 'nombre'})
    .drop_duplicates('nombre')
    .reset_index(drop=True)
)
profesor.insert(0, 'profesor_id', range(1, len(profesor) + 1))

display(profesor.head(5))
print_insert('profesor', profesor, ['nombre', 'rol_id'])

,profesor_id,nombre,rol_id
0,1,Noa Yáñez,1
1,2,Saturnina Benitez,1
2,3,Anna Feliu,1
3,4,Rosalva Ayuso,1
4,5,Ana Sofía Ferrer,1


INSERT INTO profesor (nombre, rol_id) VALUES
    ('Noa Yáñez', 1),
    ('Saturnina Benitez', 1),
    ('Anna Feliu', 1),
    ('Rosalva Ayuso', 1),
    ('Ana Sofía Ferrer', 1),
    ('Angélica Corral', 1),
    ('Ariel Lledó', 1),
    ('Mario Prats', 2),
    ('Luis Ángel Suárez', 2),
    ('María Dolores Diaz', 2);



## 8. Profesor_grupo

In [ ]:
profesor_grupo = (
    claustro
    .merge(profesor.rename(columns={'nombre': 'Nombre'}), on='Nombre')
    .merge(vertical,  left_on='Vertical',           right_on='vertical')
    .merge(promocion, left_on=['Promocion', 'Anio'], right_on=['nombre', 'anio'])
    .merge(campus,    left_on='Campus',              right_on='campus')
    .merge(modalidad, left_on='Modalidad',           right_on='modalidad')
    .merge(
        grupo[['grupo_id', 'campus_id', 'vertical_id', 'promocion_id', 'modalidad_id']],
        on=['campus_id', 'vertical_id', 'promocion_id', 'modalidad_id']
    )
    [['profesor_id', 'grupo_id']]
    .sort_values(['grupo_id', 'profesor_id'])
    .reset_index(drop=True)
)
profesor_grupo.insert(0, 'profesor_grupo_id', range(1, len(profesor_grupo) + 1))

display(profesor_grupo.head(20))
print_insert('profesor_grupo', profesor_grupo, ['profesor_id', 'grupo_id'])

,profesor_grupo_id,profesor_id,grupo_id
0,1,1,1
1,2,2,1
2,3,7,1
3,4,10,2
4,5,3,4
5,6,9,5
6,7,6,6
7,8,4,7
8,9,5,8
9,10,8,9


INSERT INTO profesor_grupo (profesor_id, grupo_id) VALUES
    (1, 1),
    (2, 1),
    (7, 1),
    (10, 2),
    (3, 4),
    (9, 5),
    (6, 6),
    (4, 7),
    (5, 8),
    (8, 9);

